# 01 — Problem & Data Design

## Default definition

A loan is classified as:

- `0` — settled within original loan duration
- `1` — not settled within original loan duration

The prediction point is approximately one-third of the original loan lifecycle.

The main constraint is simple: **the model may only use information available by the prediction point.**

## Synthetic data

The public dataset is synthetic but follows the structure of the real analytical problem:

- multiple repayment frequencies
- loan-level and transaction-level data
- early missed-payment behavior
- consecutive misses and recovery
- overdue persistence
- prior borrowing history
- product, sector and geographic heterogeneity
- noisy / exceptional observations
- imbalanced eventual outcomes

The synthetic generator is not intended to reproduce proprietary records or exact production distributions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N_BORROWERS = 12000
N_LOANS = 30000

OUTPUT_DIR = Path("../data/synthetic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng

## Borrowers

In [ ]:
borrowers = pd.DataFrame({
    "borrower_id": np.arange(1, N_BORROWERS + 1),
    "risk_propensity": rng.normal(0, 1, N_BORROWERS),
    "prior_loan_count": np.clip(rng.poisson(2.5, N_BORROWERS), 0, 12),
    "region": rng.choice(
        ["Region_A", "Region_B", "Region_C", "Region_D", "Region_E"],
        N_BORROWERS,
        p=[0.24, 0.22, 0.20, 0.18, 0.16]
    ),
    "sector": rng.choice(
        ["Agriculture", "Trade", "Services", "Other"],
        N_BORROWERS,
        p=[0.30, 0.32, 0.23, 0.15]
    ),
})

borrowers.head()

## Loans

In [ ]:
frequency_days = {
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28,
}

frequency = rng.choice(
    list(frequency_days),
    N_LOANS,
    p=[0.50, 0.30, 0.20]
)

loan_duration_days = np.select(
    [
        frequency == "Weekly",
        frequency == "Bi-weekly",
        frequency == "Monthly",
    ],
    [
        rng.choice([84, 98, 112, 140], N_LOANS),
        rng.choice([84, 112, 140, 168], N_LOANS),
        rng.choice([112, 140, 168, 196, 224], N_LOANS),
    ],
)

loan_borrower = rng.integers(1, N_BORROWERS + 1, N_LOANS)

loans = pd.DataFrame({
    "loan_id": np.arange(1, N_LOANS + 1),
    "borrower_id": loan_borrower,
    "frequency_name": frequency,
    "original_loan_duration_days": loan_duration_days,
    "disbursed_amount": np.round(
        np.exp(rng.normal(np.log(30000), 0.55, N_LOANS)), 0
    ),
    "interest_rate": np.round(
        np.clip(rng.normal(0.22, 0.05, N_LOANS), 0.08, 0.40), 4
    ),
    "product_group": rng.choice(
        ["Product_A", "Product_B", "Product_C", "Product_D"],
        N_LOANS,
        p=[0.35, 0.30, 0.20, 0.15]
    ),
})

loans = loans.merge(
    borrowers,
    on="borrower_id",
    how="left"
)

loans["installment_days"] = loans["frequency_name"].map(frequency_days)
loans["original_no_of_installments"] = (
    loans["original_loan_duration_days"] // loans["installment_days"]
).astype(int)

loans["installment_amount"] = np.round(
    loans["disbursed_amount"] * (1 + loans["interest_rate"])
    / loans["original_no_of_installments"],
    0
)

loans.head()

## Repayment transactions

In [ ]:
# Generate a compact scheduled-installment table.
schedule_parts = []

for row in loans.itertuples(index=False):
    n = int(row.original_no_of_installments)

    schedule_parts.append(
        pd.DataFrame({
            "loan_id": row.loan_id,
            "borrower_id": row.borrower_id,
            "installment_no": np.arange(1, n + 1),
            "installment_days": row.installment_days,
            "installment_amount": row.installment_amount,
            "risk_propensity": row.risk_propensity,
        })
    )

transactions = pd.concat(schedule_parts, ignore_index=True)

transactions["early_period"] = (
    transactions["installment_no"]
    <= transactions.groupby("loan_id")["installment_no"].transform("max") / 3
)

risk = 1 / (1 + np.exp(-transactions["risk_propensity"]))

# Risk is allowed to influence early misses; later behavior is also noisy.
miss_prob = np.clip(
    0.035 + 0.22 * risk + 0.06 * transactions["early_period"],
    0.02, 0.55
)

transactions["missed_installment"] = (
    rng.random(len(transactions)) < miss_prob
).astype(int)

transactions["recovery_delay_cycles"] = 0

miss_idx = transactions["missed_installment"].eq(1)
transactions.loc[miss_idx, "recovery_delay_cycles"] = rng.choice(
    [1, 2, 3, 4],
    miss_idx.sum(),
    p=[0.50, 0.28, 0.15, 0.07]
)

transactions["overdue_days"] = (
    transactions["recovery_delay_cycles"]
    * transactions["installment_days"]
)

transactions["consecutive_missed_so_far"] = (
    transactions.groupby("loan_id")["missed_installment"]
    .transform(
        lambda s: s.groupby((s == 0).cumsum()).cumsum()
    )
)

transactions.head()

## Add a small amount of operational noise

In [ ]:
# These are synthetic anomalies, not real production rules.
anomaly_rng = rng.random(len(transactions))

transactions.loc[anomaly_rng < 0.002, "overdue_days"] *= 0
transactions.loc[
    (anomaly_rng >= 0.002) & (anomaly_rng < 0.004),
    "recovery_delay_cycles"
] = transactions.loc[
    (anomaly_rng >= 0.002) & (anomaly_rng < 0.004),
    "recovery_delay_cycles"
].clip(upper=1)

transactions["overdue_days"] = transactions["overdue_days"].astype(int)

transactions.shape

## Eventual outcome

In [ ]:
full_behavior = (
    transactions.groupby("loan_id", as_index=False)
    .agg(
        total_missed_installments=("missed_installment", "sum"),
        max_consecutive_missed=("consecutive_missed_so_far", "max"),
        max_overdue_days=("overdue_days", "max"),
        total_overdue_days=("overdue_days", "sum"),
        total_recovery_delay_cycles=("recovery_delay_cycles", "sum"),
    )
)

loans = loans.merge(full_behavior, on="loan_id", how="left")

# The hidden generating process creates an eventual outcome.
default_score = (
    -2.35
    + 0.16 * loans["total_missed_installments"]
    + 0.20 * loans["max_consecutive_missed"]
    + 0.007 * loans["max_overdue_days"]
    + 0.18 * loans["risk_propensity"]
    + 0.04 * loans["sector"].isin(["Trade", "Other"]).astype(int)
)

p_default = 1 / (1 + np.exp(-default_score))
loans["is_good_or_bad"] = (rng.random(N_LOANS) < p_default).astype(int)

loans["prediction_installment"] = np.ceil(
    loans["original_no_of_installments"] / 3
).astype(int)

loans[["loan_id", "prediction_installment", "is_good_or_bad"]].head()

## Early-life modeling dataset

In [ ]:
early_transactions = transactions.merge(
    loans[["loan_id", "prediction_installment"]],
    on="loan_id",
    how="left"
)

early_transactions = early_transactions[
    early_transactions["installment_no"]
    <= early_transactions["prediction_installment"]
].copy()

early_features = (
    early_transactions.groupby("loan_id", as_index=False)
    .agg(
        early_missed_installment_count=("missed_installment", "sum"),
        early_max_consecutive_missed=("consecutive_missed_so_far", "max"),
        early_max_overdue_days=("overdue_days", "max"),
        early_total_overdue_days=("overdue_days", "sum"),
        early_recovery_delay_cycles=("recovery_delay_cycles", "sum"),
    )
)

modeling_base = loans[[
    "loan_id",
    "borrower_id",
    "frequency_name",
    "product_group",
    "sector",
    "region",
    "original_loan_duration_days",
    "original_no_of_installments",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "prediction_installment",
    "is_good_or_bad",
]].merge(
    early_features,
    on="loan_id",
    how="left"
)

modeling_base["pass_due_cycle_ratio"] = (
    modeling_base["early_max_overdue_days"]
    / modeling_base["installment_days"]
    if "installment_days" in modeling_base.columns
    else modeling_base["early_max_overdue_days"]
    / modeling_base["frequency_name"].map(frequency_days)
)

modeling_base["overdue_amount_proxy"] = (
    modeling_base["early_missed_installment_count"]
    * modeling_base["installment_amount"]
)

modeling_base["overdue_proportion"] = (
    modeling_base["overdue_amount_proxy"]
    / (
        modeling_base["prediction_installment"]
        * modeling_base["installment_amount"]
    )
).clip(0, 1)

modeling_base["missed_installment_proportion"] = (
    modeling_base["early_missed_installment_count"]
    / modeling_base["prediction_installment"].clip(lower=1)
)

modeling_base = modeling_base.drop(columns=["risk_propensity"], errors="ignore")

modeling_base.head()

## Validation

In [ ]:
checks = {
    "loans": len(loans),
    "transactions": len(transactions),
    "modeling_rows": len(modeling_base),
    "missing_target": int(modeling_base["is_good_or_bad"].isna().sum()),
    "default_rate": round(modeling_base["is_good_or_bad"].mean(), 4),
    "frequencies": modeling_base["frequency_name"].nunique(),
    "products": modeling_base["product_group"].nunique(),
    "sectors": modeling_base["sector"].nunique(),
}

pd.Series(checks)

## Save

In [ ]:
loans.to_csv(OUTPUT_DIR / "synthetic_loans.csv", index=False)
transactions.to_csv(OUTPUT_DIR / "synthetic_transactions.csv", index=False)
modeling_base.to_csv(
    OUTPUT_DIR / "synthetic_early_modeling_base.csv",
    index=False
)

print("Saved synthetic datasets to:", OUTPUT_DIR.resolve())